# Healthcare Patient Deterioration Analytics
**Student:** Vansh Jain \
**Technologies Used:** Azur Data Factory · Databricks PySpark · Unity Catalog · Spark MLlib \
**Notebook:** Bronze Layer\
**Layers:** Bronze → Silver → Gold

### Importing Libraries

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import(
    current_timestamp,
    input_file_name,
    col
)
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

### Storage account connection setup

In [0]:
storage_account = 'healthcaremedalliondev'
storage_key = '<Your Account Key>'

In [0]:
spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)

In [0]:
raw_path = f"abfss://raw@{storage_account}.dfs.core.windows.net/"
bronze_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/bronze_patient_vitals"

### Define Schema

In [0]:
healthcare_schema = StructType([
    StructField("hour_from_admission", IntegerType(), True),
    StructField("heart_rate", DoubleType(), True),
    StructField("respiratory_rate", DoubleType(), True),
    StructField("spo2_pct", DoubleType(), True),
    StructField("temperature_c", DoubleType(), True),
    StructField("systolic_bp", DoubleType(), True),
    StructField("diastolic_bp", DoubleType(), True),
    StructField("oxygen_device", StringType(), True),
    StructField("oxygen_flow", DoubleType(), True),
    StructField("mobility_score", IntegerType(), True),
    StructField("nurse_alert", IntegerType(), True),
    StructField("wbc_count", DoubleType(), True),
    StructField("lactate", DoubleType(), True),
    StructField("creatinine", DoubleType(), True),
    StructField("crp_level", DoubleType(), True),
    StructField("hemoglobin", DoubleType(), True),
    StructField("sepsis_risk_score", DoubleType(), True),
    StructField("age", IntegerType(), True),
    StructField("gender", StringType(), True),
    StructField("comorbidity_index", IntegerType(), True),
    StructField("admission_type", StringType(), True),
    StructField("deterioration_next_12h", IntegerType(), True)
])

### Load Data

In [0]:
bronze_df = (
    spark.read
    .format('csv')
    .option('header', 'true')
    .schema(healthcare_schema)
    .load(raw_path)
)

### Basic data understanding

In [0]:
bronze_df.printSchema()

In [0]:
print(f'total records: {bronze_df.count()}')

In [0]:
display(bronze_df.limit(10))

### Generating Metadata Columns

In [0]:
bronze_df = bronze_df.withColumn(
    'ingestion_timestamp',
    current_timestamp()
)

In [0]:
bronze_df = bronze_df.withColumn(
    'source_file_name',
    col('_metadata.file_path')
)

In [0]:
bronze_df.select(
    'source_file_name',
    'ingestion_timestamp'
).show(5, truncate=False)

### Writing Data to ADLS

In [0]:
(
    bronze_df.write
    .format('delta')
    .mode('overwrite')
    .save(bronze_path)
)

### Write Validation

Validating that our data is stored at right place in right format

In [0]:
bronze_table = spark.read.format('delta').load(bronze_path)

In [0]:
print(bronze_table.count())

In [0]:
bronze_table.printSchema()

### Data Quantity Check
Validating that no data get lost during storage

In [0]:
bronze_table.select('source_file_name').distinct().count()

In [0]:
display(bronze_table)